## 1. Set up YOLOv5



In [ ]:
import os, sys, subprocess
from pathlib import Path
import torch
from IPython.display import display, Image as IPyImage

YOLOV5_DIR = Path("/content/yolov5")
if not YOLOV5_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/ultralytics/yolov5.git", str(YOLOV5_DIR)],
        check=True
    )
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(YOLOV5_DIR / "requirements.txt")],
    check=True
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Upload the trained model



In [ ]:
from google.colab import files

uploaded_weights = files.upload()
pt_files = [name for name in uploaded_weights if name.lower().endswith(".pt")]
if not pt_files:
    raise ValueError("Upload your trained model file ending in .pt (usually best.pt).")

WEIGHTS_PATH = Path("/content") / pt_files[0]
print("Model:", WEIGHTS_PATH)
print("Size (MB):", round(WEIGHTS_PATH.stat().st_size / (1024 * 1024), 2))

## 3. Upload an image



In [ ]:
uploaded_images = files.upload()
image_exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
image_files = [name for name in uploaded_images if Path(name).suffix.lower() in image_exts]
if not image_files:
    raise ValueError("Upload an image: JPG, JPEG, PNG, WEBP, or BMP.")

IMAGE_PATH = Path("/content") / image_files[0]
print("Image:", IMAGE_PATH)
display(IPyImage(filename=str(IMAGE_PATH)))

## 4. Run object detection



In [ ]:
IMG_SIZE = 416
CONFIDENCE_THRESHOLD = 0.25
DEVICE = "0" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

sys.path.insert(0, str(YOLOV5_DIR))
os.chdir(YOLOV5_DIR)

model = torch.hub.load(
    str(YOLOV5_DIR), "custom",
    path=str(WEIGHTS_PATH), source="local"
)
model.conf = CONFIDENCE_THRESHOLD
model.to(DEVICE)

results = model(str(IMAGE_PATH), size=IMG_SIZE)
detections = results.pandas().xyxy[0]

if detections.empty:
    print("No objects detected above the confidence threshold.")
else:
    display(detections[["name", "confidence", "xmin", "ymin", "xmax", "ymax"]])

RESULT_DIR = Path("/content/yolov5_inference")
RESULT_DIR.mkdir(parents=True, exist_ok=True)
results.save(save_dir=str(RESULT_DIR))
RESULT_IMAGE = RESULT_DIR / IMAGE_PATH.name

if RESULT_IMAGE.exists():
    display(IPyImage(filename=str(RESULT_IMAGE)))
    print("Annotated result:", RESULT_IMAGE)
else:
    from PIL import Image
    RESULT_IMAGE = RESULT_DIR / "prediction.jpg"
    Image.fromarray(results.render()[0]).save(RESULT_IMAGE)
    display(IPyImage(filename=str(RESULT_IMAGE)))
    print("Annotated result:", RESULT_IMAGE)

## 5. Download the result


In [ ]:
files.download(str(RESULT_IMAGE))